In [23]:
import numpy as np
import ipywidgets as widgets
import matplotlib.pyplot as plt


In [13]:
rng = np.random.default_rng(42)

In [14]:
# population_alphas - (pop_size, K)
def fitness_pop(population_alphas, lengths, target_point):
    thetas = np.cumsum(population_alphas, axis=1)

    tip_xs = np.sum(lengths * np.cos(thetas), axis=1)
    tip_ys = np.sum(lengths * np.sin(thetas), axis=1)

    dist_sq = (tip_xs - target_point[0])**2 + (tip_ys - target_point[1])**2

    return dist_sq

def init_population(pop_size, dim, bounds, rng, init_sigma=0.1):
    low, high = bounds
    x = rng.uniform(low, high, size=(pop_size, dim))
    sigma0 = init_sigma * (high - low)
    sigma = np.ones((pop_size, dim)) * sigma0
    return x, sigma

def mutate(pop_x, sigma, tau, tau0, rng, min_sigma=1e-8):
    n, d = pop_x.shape

    eps0 = rng.normal(loc=0.0, scale=tau0, size=(n, 1))
    eps = rng.normal(loc=0.0, scale=tau, size=(n, d))

    new_sigma = sigma * np.exp(eps + eps0)
    new_sigma = np.maximum(new_sigma, min_sigma)

    noise = rng.normal(loc=0.0, scale=new_sigma)
    new_x = pop_x + noise
    return new_x, new_sigma


def es(
        lengths,
        target_point,
        rng,
        bounds: np.ndarray,
        mu: int = 15,
        lambda_: int = 100,
        iters: int = 1000,
        plus: bool = True,
        tau: float | None = None,
        tau0: float | None = None,
):
    dim = len(lengths)

    if tau is None:
        tau = 1.0 / np.sqrt(2.0 * np.sqrt(dim))
    if tau0 is None:
        tau0 = 1.0 / np.sqrt(2.0 * dim)

    x, sigma = init_population(mu, dim, bounds, rng)
    fitness = fitness_pop(x, lengths, target_point)

    history = []
    history_solutions = []
    low, high = bounds

    # append any solution from an initial population
    history.append(fitness[0])
    history_solutions.append(x[0].copy())

    for _ in range(iters):
        parent_indices = rng.integers(0, mu, size=lambda_)
        parent_x = x[parent_indices]
        parent_sigma = sigma[parent_indices]

        child_x, child_sigma = mutate(parent_x, parent_sigma, tau, tau0, rng)
        child_x = np.clip(child_x, low, high)
        child_fitness = fitness_pop(child_x, lengths, target_point)

        if plus:
            all_x = np.vstack([x, child_x])
            all_sigma = np.vstack([sigma, child_sigma])
            all_fitness = np.concatenate([fitness, child_fitness])

            best_idx = np.argsort(all_fitness)[:mu]
            x = all_x[best_idx]
            sigma = all_sigma[best_idx]
            fitness = all_fitness[best_idx]
        else:
            best_idx = np.argsort(child_fitness)[:mu]
            x = child_x[best_idx]
            sigma = child_sigma[best_idx]
            fitness = child_fitness[best_idx]

        history.append(fitness[0])
        history_solutions.append(x[0].copy())

    best_idx = np.argmin(fitness)
    return x[best_idx], fitness[best_idx], np.array(history), np.array(history_solutions)


In [15]:
def get_joint_coordinates(alphas, lengths):
    thetas = np.cumsum(alphas)

    x_coords = np.concatenate(([0], np.cumsum(lengths * np.cos(thetas))))
    y_coords = np.concatenate(([0], np.cumsum(lengths * np.sin(thetas))))

    return x_coords, y_coords



def plot_robot_arm(gen_idx, lengths, history_solutions, target_point ):
    alphas = history_solutions[gen_idx]
    x_coords, y_coords = get_joint_coordinates(alphas, lengths)
    tip_x, tip_y = x_coords[-1], y_coords[-1]

    dist = np.sqrt((tip_x - target_point[0])**2 + (tip_y - target_point[1])**2)
    plt.figure(figsize=(8, 8))

    max_reach = np.sum(lengths)
    limit = np.max([max_reach * 1.2, abs(target_point[0]), abs(target_point[1])]) + 0.5
    plt.xlim(-limit, limit)
    plt.ylim(-limit, limit)


    plt.plot(x_coords, y_coords, '-o', linewidth=3, label='Robot')
    plt.scatter(tip_x, tip_y, color='pink', edgecolors='black', zorder=10, label='Tip')
    plt.scatter(target_point[0], target_point[1], color='red', s=150, marker='x')

    plt.grid(True, which='both', linestyle='--')
    plt.axhline(0, color='black', linewidth=0.5)
    plt.axvline(0, color='black', linewidth=0.5)
    plt.title(f"{"Inverse Kinematic Problem"}\nGeneration: {gen_idx}\nError: {dist:.9f}")
    plt.show()


In [16]:
def create_bounds(num_joints, angle_limit):
    return np.array([
        [angle_limit[0]] * num_joints,
        [angle_limit[1]] * num_joints
    ])

def create_alternating_bounds(n):
    lows = []
    highs = []
    for i in range(n):
        if i % 2 == 0:
            lows.append(0)
            highs.append(np.pi/1.5)
        else:
            lows.append(-np.pi/1.5)
            highs.append(0)
    return np.array([lows, highs])

DATASETS = [
    {
        "lengths": np.array([2.0, 2.0, 2.0]),
        "target": np.array([3.0, 3.0]),
        "bounds": create_bounds(3, (-np.pi, np.pi)),
    },
    {
        "lengths": np.array([1.5, 1.5, 1.5, 1.5]),
        "target": np.array([-2.0, 2.0]),
        "bounds": np.array([
            [-np.pi, -np.pi/2, -np.pi/2, -np.pi/2],
            [ np.pi,  np.pi/2,  np.pi/2,  np.pi/2]
        ]),
    },
    {
        "lengths": np.ones(10) * 0.5,
        "target": np.array([-3.0, -1.0]),
        "bounds": create_bounds(10, (-np.pi/1.5, np.pi/1.5)),
    },
    {
        "lengths": np.array([1.0, 1.0]),
        "target": np.array([5.0, 5.0]),
        "bounds": create_bounds(2, (-np.pi, np.pi)),
    },
    {
        "lengths": np.ones(15) * 1.0,
        "target": np.array([1.0, 1.0]),
        "bounds": create_bounds(15, (-np.pi, np.pi)),
    },
    {
        "lengths": np.ones(20) * 1.0,
        "target": np.array([10.0, 0.0]),
        "bounds": create_alternating_bounds(20),
    },
    {
        "lengths": np.ones(5) * 3.0,
        "target": np.array([1.0, 1.0]),
        "bounds": create_alternating_bounds(5),
    }
]

In [17]:
i = 6
problem = DATASETS[i]

print(f"Running problem {i+1}")

best_x, best_fit, history, history_solutions = es(
    lengths=problem["lengths"],
    target_point=problem["target"],
    rng=rng,
    bounds=problem["bounds"],
    iters=400
)

widgets.interact(
    plot_robot_arm,
    gen_idx=widgets.IntSlider(min=0, max=len(history_solutions)-1, value=0),
    history_solutions=widgets.fixed(history_solutions),
    lengths=widgets.fixed(problem["lengths"]),
    target_point=widgets.fixed(problem["target"])
);

Running problem 7


interactive(children=(IntSlider(value=0, description='gen_idx', max=400), Output()), _dom_classes=('widget-int…

# Zadanie 5

In [18]:
def point_segment_distance(start_pts, end_pts, point):
    AB = end_pts - start_pts
    AP = point - start_pts
    ap_dot_ab = np.sum(AP * AB, axis=2)
    ab_sq = np.sum(AB * AB, axis=2)

    t = ap_dot_ab / (ab_sq + 1e-8)

    t = np.clip(t, 0.0, 1.0)
    closest_points = start_pts + (AB * t[:, :, np.newaxis])
    diff = point - closest_points
    dists = np.sqrt(np.sum(diff**2, axis=2))

    return dists

def collision_penalty(xs, ys, zs, obstacles, arm_thickness=0.2):
    n_pop, K = xs.shape
    zeros = np.zeros((n_pop, 1))

    joints_x = np.hstack([zeros, xs])
    joints_y = np.hstack([zeros, ys])
    joints_z = np.hstack([zeros, zs])

    joints = np.stack([joints_x, joints_y, joints_z], axis=2)

    starts = joints[:, :-1, :]
    ends   = joints[:, 1:, :]

    total_penalty = np.zeros(n_pop)

    for obs in obstacles:
        ox, oy, oz, radius = obs
        center = np.array([ox, oy, oz])

        dists = point_segment_distance(starts, ends, center)
        threshold = radius + arm_thickness

        penetration = np.maximum(0, threshold - dists)
        penalty_per_ind = np.sum(penetration**2, axis=1)

        total_penalty += penalty_per_ind

    return total_penalty

In [19]:
def fitness_with_obstacles(population_angles, lengths, target_point, obstacles):
    n_pop, dim = population_angles.shape
    K = len(lengths)

    angles_reshaped = population_angles.reshape(n_pop, K, 2)
    abs_angles = np.cumsum(angles_reshaped, axis=1)

    thetas = abs_angles[:, :, 0]
    phis = abs_angles[:, :, 1]

    L = lengths.reshape(1, K)

    dx = L * np.sin(phis) * np.cos(thetas)
    dy = L * np.sin(phis) * np.sin(thetas)
    dz = L * np.cos(phis)

    xs = np.cumsum(dx, axis=1)
    ys = np.cumsum(dy, axis=1)
    zs = np.cumsum(dz, axis=1)

    tip_xs = xs[:, -1]
    tip_ys = ys[:, -1]
    tip_zs = zs[:, -1]

    dist_sq = (tip_xs - target_point[0]) ** 2 + \
              (tip_ys - target_point[1]) ** 2 + \
              (tip_zs - target_point[2]) ** 2


    WEIGHT = 1000
    if obstacles:
        penalty = collision_penalty(xs, ys, zs, obstacles)
        return dist_sq + (penalty * WEIGHT)

    return dist_sq

def es_3d_constrained(
        lengths,
        target_point,
        obstacles,
        rng,
        bounds: np.ndarray,
        mu: int = 15,
        lambda_: int = 100,
        iters: int = 1000,
        plus: bool = True,
        tau: float | None = None,
        tau0: float | None = None,
):
    dim = 2*len(lengths)
    if tau is None: tau = 1.0 / np.sqrt(2.0 * np.sqrt(dim))
    if tau0 is None: tau0 = 1.0 / np.sqrt(2.0 * dim)

    x, sigma = init_population(mu, dim, bounds, rng)

    fitness = fitness_with_obstacles(x, lengths, target_point, obstacles)

    history = []
    history_solutions = []
    low, high = bounds

    history.append(fitness[0])
    history_solutions.append(x[0].copy())

    for _ in range(iters):
        parent_indices = rng.integers(0, mu, size=lambda_)
        parent_x = x[parent_indices]
        parent_sigma = sigma[parent_indices]

        child_x, child_sigma = mutate(parent_x, parent_sigma, tau, tau0, rng)
        child_x = np.clip(child_x, low, high)

        child_fitness = fitness_with_obstacles(child_x, lengths, target_point, obstacles)

        if plus:
            all_x = np.vstack([x, child_x])
            all_sigma = np.vstack([sigma, child_sigma])
            all_fitness = np.concatenate([fitness, child_fitness])

            best_idx = np.argsort(all_fitness)[:mu]
            x = all_x[best_idx]
            sigma = all_sigma[best_idx]
            fitness = all_fitness[best_idx]
        else:
            best_idx = np.argsort(child_fitness)[:mu]
            x = child_x[best_idx]
            sigma = child_sigma[best_idx]
            fitness = child_fitness[best_idx]

        history.append(fitness[0])
        history_solutions.append(x[0].copy())

    best_idx = np.argmin(fitness)
    return x[best_idx], fitness[best_idx], np.array(history), np.array(history_solutions)


In [20]:
def get_joint_coordinates_3d(flat_angles, lengths):
    K = len(lengths)
    angles = flat_angles.reshape(K, 2)

    abs_angles = np.cumsum(angles, axis=0)
    thetas = abs_angles[:, 0]
    phis   = abs_angles[:, 1]

    dz = lengths * np.cos(phis)
    dx = lengths * np.sin(phis) * np.cos(thetas)
    dy = lengths * np.sin(phis) * np.sin(thetas)

    x_coords = np.concatenate(([0], np.cumsum(dx)))
    y_coords = np.concatenate(([0], np.cumsum(dy)))
    z_coords = np.concatenate(([0], np.cumsum(dz)))

    return x_coords, y_coords, z_coords


# Plotting function generated by Gemini
import plotly.graph_objects as go

def plot_robot_arm_plotly(gen_idx, lengths, history_solutions, target_point, obstacles=None):
    flat_angles = history_solutions[gen_idx]
    xs, ys, zs = get_joint_coordinates_3d(flat_angles, lengths)


    max_reach = np.sum(lengths)
    limit = max(max_reach, np.max(np.abs(target_point)), target_point[0], target_point[1], target_point[2]) * 1.1

    fig = go.Figure()

    fig.add_trace(go.Scatter3d(
        x=xs, y=ys, z=zs,
        mode='lines+markers',
        line=dict(color='blue', width=8),
        marker=dict(size=5, color='green'),
        name='Robot Arm',
        hoverinfo='skip'
    ))

    fig.add_trace(go.Scatter3d(
        x=[target_point[0]], y=[target_point[1]], z=[target_point[2]],
        mode='markers',
        marker=dict(size=3, color='red', symbol='x', line=dict(width=2, color='red')),
        name='Target',
        hoverinfo='name+x+y+z'
    ))

    fig.add_trace(go.Scatter3d(
        x=[0], y=[0], z=[0],
        mode='markers',
        marker=dict(size=3, color='black'),
        name='Base',
        hoverinfo='skip'
    ))

    if obstacles is not None:
        for i, obs in enumerate(obstacles):
            ox, oy, oz, r = obs

            # Parametric equation of a sphere
            # We create a mesh grid to draw the surface
            u = np.linspace(0, 2 * np.pi, 20)
            v = np.linspace(0, np.pi, 20)
            sx = ox + r * np.outer(np.cos(u), np.sin(v))
            sy = oy + r * np.outer(np.sin(u), np.sin(v))
            sz = oz + r * np.outer(np.ones(np.size(u)), np.cos(v))

            fig.add_trace(go.Surface(
                x=sx, y=sy, z=sz,
                opacity=0.3,          # Semi-transparent so you can see the arm inside/behind it
                colorscale='Reds',    # Red tint for danger/obstacle
                showscale=False,      # Hide the color bar
                name=f'Obstacle {i+1}',
                hoverinfo='skip'
            ))

    fig.update_layout(
        title=f"Generation {gen_idx}",
        width=800, height=800,
        autosize=False,
        showlegend=True,
        scene_dragmode='turntable',
        template="plotly_white",
        scene=dict(
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=1),
            xaxis=dict(range=[-limit, limit], autorange=False, title='X', zeroline=True),
            yaxis=dict(range=[-limit, limit], autorange=False, title='Y', zeroline=True),
            zaxis=dict(range=[-limit, limit], autorange=False, title='Z', zeroline=True),

            xaxis_showspikes=False,
            yaxis_showspikes=False,
            zaxis_showspikes=False,
        ),
    )

    fig.show()

In [21]:
K = 20
lengths = np.ones(K) * 1.5
target_3d = np.array([15.0, 15.0, 15.0])
bounds_3d = (-np.pi, np.pi)
obstacles = []

for _ in range(20):
    ox = rng.uniform(3.0, 12.0)
    oy = rng.uniform(3.0, 12.0)
    oz = rng.uniform(3.0, 12.0)

    r = rng.uniform(0.8, 2.0)

    obstacles.append((ox, oy, oz, r))

best_sol, best_fit, fit_hist, sol_hist = es_3d_constrained(
    lengths=lengths,
    target_point=target_3d,
    obstacles=obstacles,
    rng=rng,
    bounds=bounds_3d,
)

In [22]:
plot_robot_arm_plotly(
    gen_idx=-1,
    lengths=lengths,
    history_solutions=sol_hist,
    target_point=target_3d,
    obstacles=obstacles
)